# What's new in MissingPatterns.jl 0.8

[MissingPatterns.jl](https://github.com/dantebertuzzi/MissingPatterns.jl) 0.8 makes the two tests
added in 0.7 easier to read correctly, and adds one new view:

- **adjusted p-values** in `missingpredictors` — one test per column means some come out
  "significant" by chance, and the new `padj` column corrects for it;
- **printed tests** — `missingmcar(io, tbl)` and `missingpredictors(io, tbl, col)` print their
  results with a reading, and the MCAR test now says which columns it left out;
- **`missinggaps`** — whether each column's missing values are isolated holes or contiguous blocks.

0.8 also requires Julia 1.10 (the current LTS) or later.

The table is the one from
[`getting-started.ipynb`](https://colab.research.google.com/github/dantebertuzzi/MissingPatterns.jl/blob/main/notebooks/getting-started.ipynb) and
[`whats-new-0.7.ipynb`](https://colab.research.google.com/github/dantebertuzzi/MissingPatterns.jl/blob/main/notebooks/whats-new-0.7.ipynb), so the holes have known causes and every result
below can be checked against them.

Full changelog: <https://github.com/dantebertuzzi/MissingPatterns.jl/blob/main/CHANGELOG.md>

**On Colab, pick the Julia runtime first:** *Runtime ▸ Change runtime type ▸ Julia*.

In [ ]:
using Pkg
Pkg.add(["MissingPatterns", "DataFrames"])
Pkg.status("MissingPatterns")   # needs 0.8 or later

In [2]:
using MissingPatterns, DataFrames, Dates, Random

## The table

A thousand records, built with three mechanisms that do not show in the table itself:

- **income** is refused more often in the south and by respondents over 70 — missing *because
  of* values in other columns;
- **lab** and **weight** come from the same clinical visit, which was skipped more often before
  2021 — missing because of the date, but not of any measured value;
- **schooling** codes "ignored" as `99` instead of `missing`.

In [3]:
Random.seed!(20260911)
n = 1000

region    = rand(["north", "south", "east", "west"], n)
date      = Date(2019, 1, 1) .+ Day.(rand(0:2_190, n))
age       = rand(18:89, n)
schooling = rand([0, 1, 2, 3, 99], n)

income = Vector{Union{Missing,Int}}(round.(Int, 1500 .+ 120 .* (age .- 18) .+ 800 .* randn(n)))
lab    = Vector{Union{Missing,Float64}}(round.(3.5 .+ 0.9 .* randn(n); digits = 2))
weight = Vector{Union{Missing,Float64}}(round.(60 .+ 12 .* randn(n); digits = 1))

for i in 1:n
    # refusal to state income: more in the south, more among the oldest
    rand() < 0.10 + 0.25 * (region[i] == "south") + 0.20 * (age[i] > 70) && (income[i] = missing)

    # a skipped visit takes the lab result, and usually the weight, with it
    if rand() < 0.30 + 0.35 * (date[i] < Date(2021, 1, 1))
        lab[i] = missing
        rand() < 0.8 && (weight[i] = missing)
    end
end

df = DataFrame(; id = 1:n, region, date, age, schooling, income, lab, weight)
first(df, 6)

Row,id,region,date,age,schooling,income,lab,weight
,Int64,String,Date,Int64,Int64,Int64?,Float64?,Float64?
1,1,west,2022-12-30,43,1,5596,1.5,95.2
2,2,east,2024-11-09,84,3,missing,missing,50.9
3,3,west,2022-12-19,42,0,4518,missing,67.9
4,4,east,2022-10-13,64,2,missing,missing,missing
5,5,south,2020-03-27,56,0,missing,missing,missing
6,6,east,2019-10-05,85,3,8935,missing,58.6


## The MCAR test, printed — and what it left out

`missingmcar(tbl)` still returns a `NamedTuple`. Given an `IO` first, it prints the result with a
one-line reading instead. Called on the whole table, with no `columns`:

In [4]:
missingmcar(stdout, df; color = :never)

 Little's MCAR test on 6 columns (id, age, schooling, income, lab, weight)
 skipped (not real-valued): region, date
 d² = 36.15   df = 21   p = 0.021
 ▲ evidence against MCAR at α = 0.05 — the patterns differ systematically, so complete-case analysis is likely biased
 1000 rows used ┊ 6 patterns ┊ EM converged in 17 iterations


The second line is new. Little's test is defined for continuous columns, so with no `columns` it
takes every real-valued one and, until 0.8, dropped the rest without a word. The result now lists
them in `skipped`, and the printed form shows them.

Read the first line, too: what got *in* matters as much. `id` is a row number, not a measurement,
and `schooling` carries its `99` code as if it were 99 years of study. The test runs, and its
p-value means little. Name the columns, and let `missingsentinels` turn the code into absence:

In [5]:
isna = sentinelisna(missingsentinels(df))
missingmcar(stdout, df; isna, columns = [:age, :income, :lab, :weight], color = :never)

 Little's MCAR test on 4 columns (age, income, lab, weight)
 d² = 22.45   df = 11   p = 0.021
 ▲ evidence against MCAR at α = 0.05 — the patterns differ systematically, so complete-case analysis is likely biased
 1000 rows used ┊ 6 patterns ┊ EM converged in 16 iterations


MCAR is rejected at 5%: the income refusals depend on age, and the test sees it in the pattern
means. The numbers are the same `missingmcar(df; ...)` returns — `r.skipped` included.

## Adjusted p-values

`missingpredictors` runs one test per other column. At the 5% level, about one in twenty
unrelated columns comes out significant by chance, so with many columns some false positives are
expected. The new `padj` field corrects for that. With the $m$ p-values that could be computed,
sorted as $p_{(1)} \le \dots \le p_{(m)}$, Holm's (1979) step-down method — the default — gives

$$
\tilde p_{(i)} = \max_{j \le i}\ \min\bigl(1,\ (m - j + 1)\,p_{(j)}\bigr),
$$

which keeps the chance of *any* false positive below $\alpha$, like Bonferroni (Dunn, 1961), while
never rejecting less. Printed, the bars of the columns with $\tilde p < \alpha$ are drawn solid:

In [6]:
missingpredictors(stdout, df, :income; isna, color = :never)

 Missingness of income: 202 of 1000 rows (20.20%)
 column     test             p      p adj  effect              detail
 region     χ²        2.18e-08   1.53e-07    0.20  ██          south: 42.1% of rows where missing vs 20.9% where present
 age        welch t   1.82e-05   1.09e-04   +0.35  ████        mean 60 where missing vs 52.47 where present
 date       welch t      0.242      1.000   +0.09  ░           mean 2022-02-18 where missing vs 2021-12-21 where present
 id         welch t      0.428      1.000   -0.06  ░           mean 486.1 where missing vs 504.1 where present
 schooling  welch t      0.700      1.000   +0.03  ░           mean 1.509 where missing vs 1.472 where present
 lab        welch t      0.756      1.000   -0.03  ░           mean 3.429 where missing vs 3.456 where present
 weight     welch t      0.870      1.000   +0.02  ░           mean 60.32 where missing vs 60.15 where present
 2 of 7 tests significant at α = 0.05 after Holm adjustment ┊ effect: standardized mea

The two mechanisms built into `income` survive the adjustment by orders of magnitude, and nothing
else comes close: region and age are what the refusals depend on. `id`, a row number, is
unrelated, as it should be.

`adjust = :bh` uses Benjamini & Hochberg (1995) instead,

$$
\tilde p_{(i)} = \min_{j \ge i}\ \min\Bigl(1,\ \frac{m}{j}\,p_{(j)}\Bigr),
$$

which controls the expected *share* of false positives among the columns declared significant —
more powerful when there are many columns and several real effects. On the clinical visit, the
two agree on the only real predictor, the date:

In [7]:
missingpredictors(stdout, df, :lab; isna, adjust = :bh, color = :never)

 Missingness of lab: 442 of 1000 rows (44.20%)
 column     test             p      p adj  effect              detail
 date       welch t   1.69e-19   1.18e-18   -0.59  ██████      mean 2021-06-12 where missing vs 2022-06-12 where present
 schooling  welch t      0.259      0.750   +0.08  ░           mean 1.53 where missing vs 1.44 where present
 age        welch t      0.322      0.750   -0.06  ░           mean 53.26 where missing vs 54.58 where present
 income     welch t      0.575      0.864   -0.04  ░           mean 5597 where missing vs 5700 where present
 region     χ²           0.617      0.864    0.04  ░           east: 27.1% of rows where missing vs 24.4% where present
 id         welch t      0.815      0.919   -0.01  ░           mean 498.1 where missing vs 502.4 where present
 weight     welch t      0.919      0.919   -0.01  ░           mean 60.08 where missing vs 60.2 where present
 1 of 7 tests significant at α = 0.05 after Benjamini–Hochberg adjustment ┊ effect: standard

`:bonferroni` and `:none` are accepted too. The data form carries `padj` as a field, between
`pvalue` and `effect`:

In [8]:
select(DataFrame(missingpredictors(df, :lab; isna)), :column, :pvalue, :padj, :effect)

Row,column,pvalue,padj,effect
,Symbol,Float64,Float64,Float64
1,date,1.69141e-19,1.18399e-18,-0.592001
2,schooling,0.259033,1.0,0.0806608
3,age,0.321607,1.0,-0.0630959
4,income,0.575295,1.0,-0.0398509
5,region,0.61687,1.0,0.0423216
6,id,0.814952,1.0,-0.0149043
7,weight,0.918578,1.0,-0.0106036


## Isolated holes or blocks — `missinggaps`

Two columns can both be 10% missing and need opposite treatment. One lost a value here and there,
which interpolation or any imputation handles; the other lost a contiguous stretch, with nothing
observed nearby to impute from. `missinggaps` measures the *runs* of consecutive missing rows.

The clearest case is a daily series. Below, a clinic's daily case counts lose a day here and there
and then three weeks in August, when its reporting system went down; the death counts lose
scattered days only. Both columns end up with the same number of missing days, and the
per-column summary can barely tell them apart:

In [9]:
Random.seed!(3)
days   = Date(2023, 1, 1):Day(1):Date(2023, 12, 31)
cases  = Vector{Union{Missing,Int}}(rand(0:12, length(days)))
cases[rand(1:length(days), 15)] .= missing                          # a day lost here and there
cases[Date(2023, 8, 7) .<= days .<= Date(2023, 8, 27)] .= missing   # ...and three weeks offline
deaths = Vector{Union{Missing,Int}}(rand(0:2, length(days)))
deaths[rand(1:length(days), 36)] .= missing                         # only scattered days
series = DataFrame(; day = collect(days), cases, deaths)
missingsummary(series; sortby = :none, color = :never)

 column  type     missing        %  distribution
 day     Date           0    0.00%                      
 cases   Int64         35    9.59%  ▁▂▁▁▁    ▁ ▅▆▁ ▁  ▁ 
 deaths  Int64         35    9.59%  ▃ ▁▁▂▂▁ ▁▁ ▁ ▂▁▁▂▁▂ 
 70 missing of 1095 cells (6.39%) across 3 columns ┊ bins of 19 rows


In [10]:
missinggaps(series; by = :day, color = :never)

 column   missing    gaps  longest  day                        mean  │ 1   2   3–5 6–10 11–30 31–100 101+
 cases         35      15       21  2023-08-07 … 2023-08-27     2.3  │ ▄▄▄              ▅▅▅▅▅            
 deaths        35      31        3  2023-11-12 … 2023-11-14     1.1  │ ▇▇▇ ▁▁▁ ▁▁▁                       
 46 gaps across 2 columns (42 isolated) ┊ bars: share of each column's missing values lying in a gap of that length ┊ rows in order of day


Now they separate. Each bar is the share of the column's missing values lying in a gap of that
length — so a column's bars add up to all its missing values — and `cases` has most of its
missing values in one 21-day gap, while every gap in `deaths` is short. `by = :day` orders the
rows by date first, so "consecutive" means consecutive in time, and the longest gap is reported in
dates.

### Reading it against chance

Short runs happen by chance. If each row went missing independently with probability $q$, the
length $L$ of a run would be geometric,

$$
P(L = \ell) = q^{\ell - 1}(1 - q), \qquad E[L] = \frac{1}{1 - q},
$$

so `mean_length` should sit near $1/(1-q)$, and a column far above it holds blocks. On the
getting-started table, ordered by date:

In [11]:
missinggaps(df; isna, by = :date, color = :never)

 column      missing    gaps  longest  date                       mean  │ 1   2   3–5 6–10 11–30 31–100 101+
 lab             442     213       11  2019-11-30 … 2019-12-24     2.1  │ ▃▃▃ ▂▂▂ ▃▃▃ ▂▂▂▂ ▁▁▁▁▁            
 weight          351     194       11  2020-03-17 … 2020-03-30     1.8  │ ▃▃▃ ▂▂▂ ▄▄▄ ▁▁▁▁ ▁▁▁▁▁            
 schooling       206     159        5  2024-07-19 … 2024-07-25     1.3  │ ▅▅▅ ▃▃▃ ▂▂▂                       
 income          202     161        4  2023-02-28 … 2023-03-04     1.3  │ ▆▆▆ ▃▃▃ ▁▁▁                       
 id                0       0        0  —                             —  │                                   
 region            0       0        0  —                             —  │                                   
 age               0       0        0  —                             —  │                                   
 727 gaps across 7 columns (479 isolated) ┊ bars: share of each column's missing values lying in a gap of that length ┊ rows in 

The clinical visit, `lab` and `weight`, has the longest runs. Is that a block, or just a higher
rate? The visit was skipped with probability 0.65 before 2021 and 0.30 after. `missinggapstats`
returns the same numbers as data, so compare each period with its geometric expectation:

In [12]:
s = sort(df, :date)
labgaps(rows) = only(filter(r -> r.column == :lab, missinggapstats(s[rows, :]; isna)))

DataFrame(map([("before 2021", s.date .< Date(2021)), ("2021 on", s.date .>= Date(2021))]) do (period, rows)
    g = labgaps(rows)
    q = g.nmissing / count(rows)
    (; period, q = round(q; digits = 3), mean_length = round(g.mean_length; digits = 2),
       expected = round(1 / (1 - q); digits = 2), longest = g.longest)
end)

Row,period,q,mean_length,expected,longest
,String,Float64,Float64,Float64,Int64
1,before 2021,0.683,3.29,3.15,11
2,2021 on,0.324,1.5,1.48,4


Close to the expectation in both periods. The runs are longer before 2021 only because more
visits were skipped then — which is how the table was built: each visit was skipped on its own,
never a whole stretch at once. The `cases` column above is the opposite: 9.6% missing would
give a mean run of about 1.1, and its 21-day gap is not something chance produces.

`missinggaps` takes `breaks` to change the length classes (`[1, 2, 8, 31]` gives 1, 2–7, 8–30
and 31+), and `sortby` to order the columns.

## References

The methods behind the views above; the [documentation](https://dantebertuzzi.github.io/MissingPatterns.jl/stable/references/) has the full list, and
the [Methods](https://dantebertuzzi.github.io/MissingPatterns.jl/stable/methods/) page every formula.

- Holm, S. (1979). A simple sequentially rejective multiple test procedure. *Scandinavian Journal of Statistics*, 6(2), 65–70. <https://www.jstor.org/stable/4615733>
- Benjamini, Y., & Hochberg, Y. (1995). Controlling the false discovery rate: a practical and powerful approach to multiple testing. *Journal of the Royal Statistical Society: Series B*, 57(1), 289–300. <https://doi.org/10.1111/j.2517-6161.1995.tb02031.x>
- Dunn, O. J. (1961). Multiple comparisons among means. *Journal of the American Statistical Association*, 56(293), 52–64. <https://doi.org/10.1080/01621459.1961.10482090>
- Little, R. J. A. (1988). A test of missing completely at random for multivariate data with missing values. *Journal of the American Statistical Association*, 83(404), 1198–1202. <https://doi.org/10.1080/01621459.1988.10478722>
- Welch, B. L. (1947). The generalization of 'Student's' problem when several different population variances are involved. *Biometrika*, 34(1–2), 28–35. <https://doi.org/10.1093/biomet/34.1-2.28>
- Rubin, D. B. (1976). Inference and missing data. *Biometrika*, 63(3), 581–592. <https://doi.org/10.1093/biomet/63.3.581>
- Little, R. J. A., & Rubin, D. B. (2019). *Statistical Analysis with Missing Data* (3rd ed.). Wiley. <https://doi.org/10.1002/9781119482260>

## Where to go next

- [Documentation](https://dantebertuzzi.github.io/MissingPatterns.jl/stable) — every keyword of every entry point
- [`novidades-0.8.ipynb`](https://colab.research.google.com/github/dantebertuzzi/MissingPatterns.jl/blob/main/notebooks/novidades-0.8.ipynb) — the same features, in Portuguese, on a
  SUS-style table
- [`whats-new-0.7.ipynb`](https://colab.research.google.com/github/dantebertuzzi/MissingPatterns.jl/blob/main/notebooks/whats-new-0.7.ipynb) — the tests these build on
- [`getting-started.ipynb`](https://colab.research.google.com/github/dantebertuzzi/MissingPatterns.jl/blob/main/notebooks/getting-started.ipynb) — the full tour of the package